# Demo: Text-to-SQL with XiYan M-Schema (OpenRouter)

In this notebook we build a **Text-to-SQL pipeline** using the XiYan-DBDescGen framework,
powered by **Gemini via OpenRouter**.

**What you will see:**
1. How to create a small relational database (Dunder Mifflin paper company)
2. How M-Schema enriches raw schema information so an LLM can write better SQL
3. How to filter schemas so the LLM only sees relevant tables
4. How multiple SQL candidates are generated using different prompt strategies
5. How a **self-correction loop** fixes broken SQL automatically
6. How candidates are **ranked** to pick the best answer

**Pre-requisites:** Basic understanding of what a database table is (covered in the lecture).


In [ ]:
#@title Pipeline Overview { display-mode: "form" }
from IPython.display import HTML, display

display(HTML('''
<div style="font-family:'Segoe UI',system-ui,-apple-system,sans-serif;max-width:920px;margin:20px auto;">

  <!-- Title -->
  <div style="text-align:center;margin-bottom:28px;">
    <div style="font-size:22px;font-weight:800;color:#0f172a;letter-spacing:-0.5px;">XiYan-SQL Pipeline Architecture</div>
    <div style="font-size:13px;color:#64748b;margin-top:4px;">From natural language question to verified SQL answer</div>
  </div>

  <!-- Row 1: Input -->
  <div style="display:flex;justify-content:center;margin-bottom:6px;">
    <div style="background:linear-gradient(135deg,#0f172a,#1e293b);color:white;padding:14px 32px;
                border-radius:14px;font-size:15px;font-weight:600;text-align:center;
                box-shadow:0 4px 12px rgba(15,23,42,0.25);">
      &ldquo;How many red papers are in stock?&rdquo;
      <div style="font-size:11px;opacity:0.6;margin-top:3px;font-weight:400;">Natural Language Question</div>
    </div>
  </div>

  <!-- Arrow -->
  <div style="text-align:center;font-size:22px;color:#94a3b8;line-height:1;">&#9660;</div>

  <!-- Stage 1: Schema Enrichment -->
  <div style="background:linear-gradient(135deg,#eef2ff,#e0e7ff);border:2px solid #a5b4fc;border-radius:16px;
              padding:18px 22px;margin-bottom:6px;position:relative;">
    <div style="display:flex;align-items:center;gap:10px;margin-bottom:12px;">
      <span style="background:#6366f1;color:white;font-size:11px;font-weight:700;padding:3px 10px;
                   border-radius:20px;">STAGE 1</span>
      <span style="font-size:15px;font-weight:700;color:#312e81;">Schema Enrichment &mdash; M-Schema</span>
      <span style="font-size:11px;color:#6366f1;margin-left:auto;font-weight:500;">&sect;2 in notebook</span>
    </div>
    <div style="display:flex;gap:10px;flex-wrap:wrap;">
      <div style="flex:1;min-width:160px;background:white;border-radius:10px;padding:12px 14px;
                  border:1px solid #c7d2fe;">
        <div style="font-size:12px;font-weight:700;color:#4338ca;">Field Classification</div>
        <div style="font-size:11px;color:#64748b;margin-top:4px;">LLM labels each column:<br>
             <span style="background:#6366f1;color:white;padding:1px 6px;border-radius:8px;font-size:10px;">Code</span>
             <span style="background:#f59e0b;color:white;padding:1px 6px;border-radius:8px;font-size:10px;">Enum</span>
             <span style="background:#10b981;color:white;padding:1px 6px;border-radius:8px;font-size:10px;">Measure</span>
             <span style="background:#ef4444;color:white;padding:1px 6px;border-radius:8px;font-size:10px;">Text</span>
        </div>
      </div>
      <div style="display:flex;align-items:center;font-size:18px;color:#a5b4fc;">&rarr;</div>
      <div style="flex:1;min-width:160px;background:white;border-radius:10px;padding:12px 14px;
                  border:1px solid #c7d2fe;">
        <div style="font-size:12px;font-weight:700;color:#4338ca;">Description Generation</div>
        <div style="font-size:11px;color:#64748b;margin-top:4px;">LLM writes natural-language descriptions for every table &amp; column</div>
      </div>
      <div style="display:flex;align-items:center;font-size:18px;color:#a5b4fc;">&rarr;</div>
      <div style="flex:1;min-width:140px;background:white;border-radius:10px;padding:12px 14px;
                  border:1px solid #c7d2fe;">
        <div style="font-size:12px;font-weight:700;color:#4338ca;">Enriched M-Schema</div>
        <div style="font-size:11px;color:#64748b;margin-top:4px;">Types + descriptions + examples &mdash; far richer than raw DDL</div>
      </div>
    </div>
  </div>

  <!-- Arrow -->
  <div style="text-align:center;font-size:22px;color:#94a3b8;line-height:1;">&#9660;</div>

  <!-- Stage 2: Schema Filtering -->
  <div style="background:linear-gradient(135deg,#f0fdf4,#dcfce7);border:2px solid #86efac;border-radius:16px;
              padding:18px 22px;margin-bottom:6px;">
    <div style="display:flex;align-items:center;gap:10px;margin-bottom:12px;">
      <span style="background:#16a34a;color:white;font-size:11px;font-weight:700;padding:3px 10px;
                   border-radius:20px;">STAGE 2</span>
      <span style="font-size:15px;font-weight:700;color:#14532d;">Schema Filtering</span>
      <span style="font-size:11px;color:#16a34a;margin-left:auto;font-weight:500;">&sect;3 in notebook</span>
    </div>
    <div style="display:flex;gap:10px;flex-wrap:wrap;">
      <div style="flex:1;min-width:180px;background:white;border-radius:10px;padding:12px 14px;
                  border:1px solid #bbf7d0;">
        <div style="font-size:12px;font-weight:700;color:#15803d;">Keyword Extraction</div>
        <div style="font-size:11px;color:#64748b;margin-top:4px;">LLM pulls key terms from the question</div>
      </div>
      <div style="display:flex;align-items:center;font-size:18px;color:#86efac;">&rarr;</div>
      <div style="flex:1;min-width:180px;background:white;border-radius:10px;padding:12px 14px;
                  border:1px solid #bbf7d0;">
        <div style="font-size:12px;font-weight:700;color:#15803d;">Schema Matching</div>
        <div style="font-size:11px;color:#64748b;margin-top:4px;">Match keywords against table names, columns, values, descriptions</div>
      </div>
      <div style="display:flex;align-items:center;font-size:18px;color:#86efac;">&rarr;</div>
      <div style="flex:1;min-width:140px;background:white;border-radius:10px;padding:12px 14px;
                  border:1px solid #bbf7d0;">
        <div style="font-size:12px;font-weight:700;color:#15803d;">Filtered Schema</div>
        <div style="font-size:11px;color:#64748b;margin-top:4px;">Only relevant tables &amp; columns sent to LLM</div>
      </div>
    </div>
  </div>

  <!-- Arrow -->
  <div style="text-align:center;font-size:22px;color:#94a3b8;line-height:1;">&#9660;</div>

  <!-- Stage 3: Multi-Candidate Generation -->
  <div style="background:linear-gradient(135deg,#faf5ff,#f3e8ff);border:2px solid #c4b5fd;border-radius:16px;
              padding:18px 22px;margin-bottom:6px;">
    <div style="display:flex;align-items:center;gap:10px;margin-bottom:12px;">
      <span style="background:#7c3aed;color:white;font-size:11px;font-weight:700;padding:3px 10px;
                   border-radius:20px;">STAGE 3</span>
      <span style="font-size:15px;font-weight:700;color:#3b0764;">Multi-Candidate SQL Generation</span>
      <span style="font-size:11px;color:#7c3aed;margin-left:auto;font-weight:500;">&sect;4 in notebook</span>
    </div>
    <div style="display:flex;gap:12px;flex-wrap:wrap;">
      <div style="flex:1;min-width:130px;background:white;border-radius:10px;padding:12px 14px;
                  border:1px solid #ddd6fe;text-align:center;">
        <div style="font-size:20px;">&#9889;</div>
        <div style="font-size:12px;font-weight:700;color:#5b21b6;">Direct</div>
        <div style="font-size:10px;color:#64748b;margin-top:2px;">Simple one-shot prompt</div>
      </div>
      <div style="flex:1;min-width:130px;background:white;border-radius:10px;padding:12px 14px;
                  border:1px solid #ddd6fe;text-align:center;">
        <div style="font-size:20px;">&#129504;</div>
        <div style="font-size:12px;font-weight:700;color:#5b21b6;">Chain-of-Thought</div>
        <div style="font-size:10px;color:#64748b;margin-top:2px;">Step-by-step reasoning</div>
      </div>
      <div style="flex:1;min-width:130px;background:white;border-radius:10px;padding:12px 14px;
                  border:1px solid #ddd6fe;text-align:center;">
        <div style="font-size:20px;">&#128270;</div>
        <div style="font-size:12px;font-weight:700;color:#5b21b6;">With Evidence</div>
        <div style="font-size:10px;color:#64748b;margin-top:2px;">Hints from M-Schema values</div>
      </div>
    </div>
  </div>

  <!-- Arrow -->
  <div style="text-align:center;font-size:22px;color:#94a3b8;line-height:1;">&#9660;</div>

  <!-- Stage 4: Self-Correction -->
  <div style="background:linear-gradient(135deg,#fff7ed,#ffedd5);border:2px solid #fdba74;border-radius:16px;
              padding:18px 22px;margin-bottom:6px;">
    <div style="display:flex;align-items:center;gap:10px;margin-bottom:12px;">
      <span style="background:#ea580c;color:white;font-size:11px;font-weight:700;padding:3px 10px;
                   border-radius:20px;">STAGE 4</span>
      <span style="font-size:15px;font-weight:700;color:#431407;">Self-Correction</span>
      <span style="font-size:11px;color:#ea580c;margin-left:auto;font-weight:500;">&sect;5 in notebook</span>
    </div>
    <div style="display:flex;align-items:center;gap:14px;flex-wrap:wrap;">
      <div style="background:white;border-radius:10px;padding:12px 16px;border:1px solid #fed7aa;flex:1;min-width:160px;">
        <div style="font-size:12px;font-weight:700;color:#c2410c;">Execute SQL</div>
        <div style="font-size:11px;color:#64748b;margin-top:3px;">Run each candidate against the database</div>
      </div>
      <div style="display:flex;align-items:center;font-size:18px;color:#fdba74;">&rarr;</div>
      <div style="background:white;border-radius:10px;padding:12px 16px;border:1px solid #fed7aa;flex:1;min-width:180px;">
        <div style="font-size:12px;font-weight:700;color:#c2410c;">Error? Feed back to LLM</div>
        <div style="font-size:11px;color:#64748b;margin-top:3px;">Send SQL + error message &rarr; get corrected query</div>
      </div>
      <div style="font-size:22px;color:#f97316;">&#128260;</div>
      <div style="background:white;border-radius:10px;padding:8px 14px;border:1px dashed #fdba74;
                  font-size:11px;color:#9a3412;font-style:italic;">Retry up to N times</div>
    </div>
  </div>

  <!-- Arrow -->
  <div style="text-align:center;font-size:22px;color:#94a3b8;line-height:1;">&#9660;</div>

  <!-- Stage 5: Selection -->
  <div style="background:linear-gradient(135deg,#fef2f2,#fee2e2);border:2px solid #fca5a5;border-radius:16px;
              padding:18px 22px;margin-bottom:6px;">
    <div style="display:flex;align-items:center;gap:10px;margin-bottom:12px;">
      <span style="background:#dc2626;color:white;font-size:11px;font-weight:700;padding:3px 10px;
                   border-radius:20px;">STAGE 5</span>
      <span style="font-size:15px;font-weight:700;color:#450a0a;">Candidate Selection</span>
      <span style="font-size:11px;color:#dc2626;margin-left:auto;font-weight:500;">&sect;6 in notebook</span>
    </div>
    <div style="display:flex;gap:12px;flex-wrap:wrap;">
      <div style="flex:1;min-width:180px;background:white;border-radius:10px;padding:12px 14px;
                  border:1px solid #fecaca;text-align:center;">
        <div style="font-size:20px;">&#128499;</div>
        <div style="font-size:12px;font-weight:700;color:#b91c1c;">Majority Vote</div>
        <div style="font-size:10px;color:#64748b;margin-top:2px;">Most common result wins</div>
      </div>
      <div style="flex:1;min-width:180px;background:white;border-radius:10px;padding:12px 14px;
                  border:1px solid #fecaca;text-align:center;">
        <div style="font-size:20px;">&#9878;</div>
        <div style="font-size:12px;font-weight:700;color:#b91c1c;">LLM-as-Judge</div>
        <div style="font-size:10px;color:#64748b;margin-top:2px;">LLM evaluates SQL quality &amp; picks best</div>
      </div>
    </div>
  </div>

  <!-- Arrow -->
  <div style="text-align:center;font-size:22px;color:#94a3b8;line-height:1;">&#9660;</div>

  <!-- Output -->
  <div style="display:flex;justify-content:center;">
    <div style="background:linear-gradient(135deg,#10b981,#059669);color:white;padding:14px 32px;
                border-radius:14px;font-size:15px;font-weight:600;text-align:center;
                box-shadow:0 4px 12px rgba(16,185,129,0.3);">
      <code style="background:rgba(255,255,255,0.2);padding:2px 8px;border-radius:6px;">SELECT SUM(stock_qty) FROM paper_products WHERE color = 'red'</code>
      <div style="font-size:13px;font-weight:400;margin-top:6px;opacity:0.85;">&#10003; Verified SQL Answer &rarr; <strong>75</strong></div>
    </div>
  </div>

  <!-- Legend -->
  <div style="margin-top:20px;padding:12px 16px;background:#f8fafc;border-radius:10px;border:1px solid #e2e8f0;
              display:flex;gap:16px;flex-wrap:wrap;align-items:center;justify-content:center;">
    <span style="font-size:11px;color:#64748b;font-weight:600;">KEY:</span>
    <span style="font-size:11px;color:#64748b;">&#129302; = LLM call</span>
    <span style="font-size:11px;color:#64748b;">&#128187; = Code logic</span>
    <span style="font-size:11px;color:#64748b;">&#128260; = Feedback loop</span>
    <span style="font-size:11px;color:#64748b;">&#128499; = Ensemble</span>
  </div>

</div>
'''))


## Section 0 — Setup & Installation

We install the required packages and import the XiYan-DBDescGen modules.

In [1]:
#@title Install dependencies & clone repos { display-mode: "form" }

import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import userdata

    # ── Clone the course repo from GitHub (public repo) ──
    REPO_URL = "https://github.com/MAS-AID-AI-Project/weekend4_agentic_public.git"
    if not os.path.exists('/content/weekend4_agentic_public'):
        !git clone {REPO_URL} /content/weekend4_agentic_public
    else:
        !cd /content/weekend4_agentic_public && git pull

    # Navigate to the project directory
    PROJECT_DIR = '/content/weekend4_agentic_public/project_saturday'
    os.chdir(PROJECT_DIR)
    if PROJECT_DIR not in sys.path:
        sys.path.insert(0, PROJECT_DIR)

    # ── Install packages ──
    !pip install -q llama-index-core llama-index-llms-openai-like sqlalchemy pandas

    # ── Clone XiYan-DBDescGen if not already in the repo ──
    if not os.path.isdir('../XiYan-DBDescGen'):
        !git clone https://github.com/XGenerationLab/XiYan-DBDescGen.git ../XiYan-DBDescGen

    # ── Load API key from Colab Secrets ──
    os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')

    print('Colab setup complete!')
else:
    print('Running locally — skipping Colab-specific setup.')

In [2]:
#@title Imports & XiYan path setup { display-mode: "form" }
import os
import sys
import sqlite3
import pandas as pd
from sqlalchemy import create_engine

# Add XiYan-DBDescGen to the Python path
XIYAN_PATH = os.path.join(os.getcwd(), "../XiYan-DBDescGen")
if os.path.isdir(XIYAN_PATH):
    sys.path.insert(0, XIYAN_PATH)

print("Setup complete.")

Setup complete.


In [3]:
#@title Patch XiYan prompts to English { display-mode: "form" }
# ── Patch XiYan prompts to output in English ──
# The default prompts in XiYan-DBDescGen are written in Chinese.
# We override them here with English equivalents so all output is in English.
#
# IMPORTANT: components.py uses `from default_prompts import X` which creates
# local bindings. We must patch BOTH default_prompts AND components modules.

from llama_index.core.prompts import PromptTemplate
from llama_index.core.prompts.prompt_type import PromptType
import default_prompts
import components

# ── English prompt definitions ──

IS_DATE_TIME_PROMPT = PromptTemplate(
    """You are a data analyst. Given the following information about a column in a database table, determine whether this column represents a date or time value.
A date/time type is composed of one or more of: year, month, day, hour, minute, second.

Answer only "yes" or "no".

{field_info_str}
""", prompt_type=PromptType.CUSTOM)

DATE_TIME_MIN_GRAN_PROMPT = PromptTemplate(
    """You are a data analyst. Given a database column that represents a date or time value, determine its minimum time granularity based on the format and example values.

Possible granularities: YEAR, MONTH, DAY, WEEK, QUARTER, HOUR, MINUTE, SECOND, MILLISECOND, MICROSECOND, OTHER

Answer with just the granularity name.

{field_info_str}
Minimum granularity: """, prompt_type=PromptType.CUSTOM)

STRING_CATEGORY_PROMPT = PromptTemplate(
    """You are a data analyst. Given the following column information, classify it as "enum", "code", or "text". Answer only one word.

- enum: values come from a small, fixed set (e.g. status, type, category)
- code: structured identifiers with a pattern (e.g. user ID, postal code)
- text: free-form text of varying length (e.g. descriptions, comments)

{field_info_str}
""", prompt_type=PromptType.CUSTOM)

NUMBER_CATEGORY_PROMPT = PromptTemplate(
    """You are a data analyst. Given the following column information, classify it as "enum", "code", or "measure". Answer only one word.

- enum: values come from a small, fixed set (e.g. status flags like 0/1)
- code: structured numeric identifiers (e.g. user ID, product ID)
- measure: a numeric metric that can be aggregated (e.g. price, quantity, amount)

{field_info_str}
""", prompt_type=PromptType.CUSTOM)

UNKNOWN_FIELD_PROMPT = PromptTemplate(
    """You are a data analyst. Given the following column information, classify it as "enum", "measure", "code", or "text". Answer only one word.

- enum: values come from a small, fixed set
- code: structured identifiers with a pattern
- measure: a numeric metric that can be aggregated
- text: free-form text

{field_info_str}
""", prompt_type=PromptType.CUSTOM)

UNDERSTAND_DB_PROMPT = PromptTemplate(
    """You are a data analyst. Given the following database schema, provide a brief summary (in English) of what domain this database covers and what kind of data it stores. Give an overall summary, not a per-table breakdown.

{db_mschema}
""", prompt_type=PromptType.CUSTOM)

DOMAIN_KNOWLEDGE_PROMPT = PromptTemplate(
    """Here is a database with the following summary:
{db_info}

Based on your knowledge, what are the typical dimensions and metrics that people care about in this domain? Answer in English.
""", prompt_type=PromptType.CUSTOM)

UNDERSTAND_FIELDS_PROMPT = PromptTemplate(
    """You are a data analyst. Here is some context about a database:

Database Info:
{db_info}

The table "{table_name}" has the following schema and sample data:
{table_mschema}

SQL:
{sql}
Examples:
{sql_res}

The fields {fields} are all classified as {category} fields. Please analyze the relationships and differences between these fields. Answer in English.
""", prompt_type=PromptType.CUSTOM)

SQL_GEN_PROMPT = PromptTemplate(
    """You are a {dialect} data analyst. Given the following database schema:

Database Schema:
{db_mschema}

User Question:
{question}
Reference Information:
{evidence}

Read and understand the database carefully. Based on the user question and reference information, generate an executable SQL query to answer the question. Wrap the SQL in ```sql and ```.
""", prompt_type=PromptType.CUSTOM)

# ── Patch both modules so the new prompts are actually used ──

# Patch default_prompts (for any code that reads from the module directly)
default_prompts.DEFAULT_IS_DATE_TIME_FIELD_PROMPT = IS_DATE_TIME_PROMPT
default_prompts.DEFAULT_DATE_TIME_MIN_GRAN_PROMPT = DATE_TIME_MIN_GRAN_PROMPT
default_prompts.DEFAULT_STRING_CATEGORY_FIELD_PROMPT = STRING_CATEGORY_PROMPT
default_prompts.DEFAULT_NUMBER_CATEGORY_FIELD_PROMPT = NUMBER_CATEGORY_PROMPT
default_prompts.DEFAULT_UNKNOWN_FIELD_PROMPT = UNKNOWN_FIELD_PROMPT
default_prompts.DEFAULT_UNDERSTAND_DATABASE_PROMPT = UNDERSTAND_DB_PROMPT
default_prompts.DEFAULT_GET_DOMAIN_KNOWLEDGE_PROMPT = DOMAIN_KNOWLEDGE_PROMPT
default_prompts.DEFAULT_UNDERSTAND_FIELDS_BY_CATEGORY_PROMPT = UNDERSTAND_FIELDS_PROMPT
default_prompts.DEFAULT_SQL_GEN_PROMPT = SQL_GEN_PROMPT

# Patch components (the actual consumer — has its own local references from `from X import Y`)
components.DEFAULT_IS_DATE_TIME_FIELD_PROMPT = IS_DATE_TIME_PROMPT
components.DEFAULT_DATE_TIME_MIN_GRAN_PROMPT = DATE_TIME_MIN_GRAN_PROMPT
components.DEFAULT_STRING_CATEGORY_FIELD_PROMPT = STRING_CATEGORY_PROMPT
components.DEFAULT_NUMBER_CATEGORY_FIELD_PROMPT = NUMBER_CATEGORY_PROMPT
components.DEFAULT_UNKNOWN_FIELD_PROMPT = UNKNOWN_FIELD_PROMPT
components.DEFAULT_UNDERSTAND_DATABASE_PROMPT = UNDERSTAND_DB_PROMPT
components.DEFAULT_GET_DOMAIN_KNOWLEDGE_PROMPT = DOMAIN_KNOWLEDGE_PROMPT
components.DEFAULT_UNDERSTAND_FIELDS_BY_CATEGORY_PROMPT = UNDERSTAND_FIELDS_PROMPT
components.DEFAULT_SQL_GEN_PROMPT = SQL_GEN_PROMPT

# ── Fix the 是/否 logic ──
# components.field_category() checks `if is_date_time == '是'` (Chinese for "yes").
# Our English prompt returns "yes"/"no", so we monkey-patch the function.

_original_field_category = components.field_category

def _english_field_category(field_type_cate, type_engine, llm=None, field_info_str=''):
    """Wrapper that normalizes yes/no responses before the original logic runs."""
    from components import call_llm

    code_res = {"category": type_engine.field_category_code_label,
                "dim_or_meas": type_engine.dimension_label}
    enum_res = {"category": type_engine.field_category_enum_label,
                "dim_or_meas": type_engine.dimension_label}
    date_res = {"category": type_engine.field_category_date_label,
                "dim_or_meas": type_engine.dimension_label}
    measure_res = {"category": type_engine.field_category_measure_label,
                   "dim_or_meas": type_engine.measure_label}
    text_res = {"category": type_engine.field_category_text_label,
                "dim_or_meas": type_engine.dimension_label}

    if field_type_cate == type_engine.field_type_date_label:
        return date_res
    elif field_type_cate == type_engine.field_type_bool_label:
        return enum_res
    else:
        kwargs = {"llm": llm, "field_info_str": field_info_str}
        is_date_time = call_llm(
            components.DEFAULT_IS_DATE_TIME_FIELD_PROMPT, **kwargs
        ).strip().lower()
        # Accept both English "yes" and Chinese "是"
        if is_date_time in ('yes', '是'):
            return date_res
        else:
            if field_type_cate == type_engine.field_type_string_label:
                res = call_llm(
                    components.DEFAULT_STRING_CATEGORY_FIELD_PROMPT, **kwargs
                ).strip().lower()
                if res == 'enum':
                    return enum_res
                elif res == 'text':
                    return text_res
                else:
                    return code_res
            elif field_type_cate == type_engine.field_type_number_label:
                res = call_llm(
                    components.DEFAULT_NUMBER_CATEGORY_FIELD_PROMPT, **kwargs
                ).strip().lower()
                if res == 'enum':
                    return enum_res
                elif res == 'measure':
                    return measure_res
                else:
                    return code_res
            else:
                res = call_llm(
                    components.DEFAULT_UNKNOWN_FIELD_PROMPT, **kwargs
                ).strip().lower()
                if res == 'enum':
                    return enum_res
                elif res == 'measure':
                    return measure_res
                elif res == 'text':
                    return text_res
                else:
                    return code_res

components.field_category = _english_field_category


# ── Patch get_single_field_info_str to use English labels ──
from schema_engine import SchemaEngine

def _english_field_info_str(self, table_name, field_name):
    """English version of get_single_field_info_str."""
    field_info = self._mschema.get_field_info(table_name, field_name)
    field_type = field_info.get('type', '')

    unique_num = self.get_column_unique_count(table_name, field_name)
    total_num = self.get_column_count(table_name, field_name)
    max_value = self.get_column_agg_value(table_name, field_name, field_type, 'max')
    min_value = self.get_column_agg_value(table_name, field_name, field_type, 'min')
    avg_value = self.get_column_agg_value(table_name, field_name, field_type, 'avg')
    max_len = self.get_column_agg_char_length(table_name, field_name, 'max')
    min_len = self.get_column_agg_char_length(table_name, field_name, 'min')

    comment = field_info.get('comment', '')
    primary_key = field_info.get("primary_key", False)
    nullable = field_info.get("nullable", True)

    parts = ['[Field Info]', f'Field Name: {field_name}', f'Field Type: {field_type}']
    dim_or_meas = field_info.get('dim_or_meas', '')
    unique = field_info.get('unique', False)
    if primary_key:
        unique = True

    if len(comment) > 0:
        parts.append(f'Description: {comment}')
    parts.append(f'Primary Key (or part of composite PK): {primary_key}')
    parts.append(f'UNIQUE: {unique}')
    parts.append(f'NULLABLE: {nullable}')
    date_min_gran = field_info.get('date_min_gran', None)
    if total_num >= 0:
        parts.append(f'COUNT: {total_num}')
    if unique_num >= 0:
        parts.append(f'COUNT(DISTINCT): {unique_num}')
    if max_value is not None:
        parts.append(f'MAX: {max_value}')
    if min_value is not None:
        parts.append(f'MIN: {min_value}')
    if avg_value is not None:
        parts.append(f'AVG: {avg_value}')
    if max_len >= 0:
        parts.append(f'MAX(CHAR_LENGTH): {max_len}')
    if min_len >= 0:
        parts.append(f'MIN(CHAR_LENGTH): {min_len}')
    if dim_or_meas in self._type_engine.dim_measure_labels:
        parts.append(f'Dimension/Measure: {dim_or_meas}')
    if date_min_gran is not None:
        parts.append(f'This field likely represents a date/time value. Estimated minimum granularity: {date_min_gran}')

    value_examples = self.get_column_value_examples(table_name, field_name, max_rows=10, max_str_len=30)
    if len(value_examples) > 0:
        parts.append(f"Value Examples: {value_examples}")

    return '\n'.join(parts)

SchemaEngine.get_single_field_info_str = _english_field_info_str


# ── Patch fields_category to remove remaining Chinese print statements ──

def _english_fields_category(self):
    """English version of SchemaEngine.fields_category — with progress bar."""
    from IPython.display import display, HTML
    import time as _time

    tables = self._mschema.tables
    total = sum(len(t['fields']) for t in tables.values())
    done = 0

    # Create a display handle we can update in place
    progress_handle = display(HTML(""), display_id=True)

    def _update_bar(table, field, category, done, total):
        pct = int(done / total * 100) if total else 0
        progress_handle.update(HTML(f"""
        <div style="font-family:system-ui,sans-serif;max-width:620px;">
          <div style="display:flex;justify-content:space-between;font-size:12px;color:#475569;margin-bottom:4px;">
            <span>Classifying fields... <strong>{done}</strong> / {total}</span>
            <span>{pct}%</span>
          </div>
          <div style="background:#e2e8f0;border-radius:8px;height:22px;overflow:hidden;">
            <div style="background:linear-gradient(90deg,#6366f1,#8b5cf6);height:100%;border-radius:8px;
                        width:{pct}%;transition:width 0.3s;"></div>
          </div>
          <div style="font-size:11px;color:#94a3b8;margin-top:4px;">
            {table}.<strong>{field}</strong> &rarr; <span style="color:#6366f1;font-weight:600;">{category}</span>
          </div>
        </div>"""))

    for table_name in tables.keys():
        fields = tables[table_name]['fields']
        for field_name, field_info in fields.items():
            field_type = field_info['type']
            field_type_cate = self._type_engine.field_type_cate(field_type)
            field_info_str = self.get_single_field_info_str(table_name, field_name)
            res = components.field_category(field_type_cate, self._type_engine, self._llm, field_info_str=field_info_str)

            if res['category'] == self._type_engine.field_category_date_label:
                min_gran = components.understand_date_time_min_gran(field_info_str, llm=self._llm)
                if min_gran in self._type_engine.date_time_min_grans:
                    self._mschema.set_column_property(table_name, field_name, "date_min_gran", min_gran)

            category = res['category']
            if category == self._type_engine.field_category_enum_label:
                examples = self.get_column_value_examples(table_name, field_name)
                examples = [s for s in examples if len(str(examples)) > 0]
                self._mschema.set_column_property(table_name, field_name, "examples", examples)
            self._mschema.set_column_property(table_name, field_name, "category", res['category'])
            self._mschema.set_column_property(table_name, field_name, "dim_or_meas", res['dim_or_meas'])

            done += 1
            _update_bar(table_name, field_name, res['category'], done, total)

    # Final state
    progress_handle.update(HTML(f"""
    <div style="font-family:system-ui,sans-serif;max-width:620px;">
      <div style="display:flex;justify-content:space-between;font-size:12px;color:#16a34a;margin-bottom:4px;">
        <span>All {total} fields classified!</span>
        <span>100%</span>
      </div>
      <div style="background:#e2e8f0;border-radius:8px;height:22px;overflow:hidden;">
        <div style="background:linear-gradient(90deg,#10b981,#059669);height:100%;border-radius:8px;width:100%;"></div>
      </div>
    </div>"""))

SchemaEngine.fields_category = _english_fields_category

print("All patches applied — fully English output.")

All patches applied — fully English output.


In [4]:
#@title Configure LLM — OpenRouter with Gemini { display-mode: "form" }
# Configure the LLM — OpenRouter with Gemini
from llama_index.llms.openai_like import OpenAILike

API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip()
if not API_KEY:
    raise RuntimeError(
        "Set OPENROUTER_API_KEY as an environment variable "
        "(or add it to Colab Secrets)."
    )

MODEL_NAME = "google/gemini-3-flash-preview"

llm = OpenAILike(
    model=MODEL_NAME,
    api_base="https://openrouter.ai/api/v1",
    api_key=API_KEY,
    is_chat_model=True,
    temperature=0,
    max_tokens=1024,
)

print(f"LLM ready: {MODEL_NAME} via OpenRouter")


LLM ready: qwen2.5:7b


---

## Section 1 — The Data: Dunder Mifflin Paper Company

Remember Dunder Mifflin from the lecture? They sell different kinds of paper through a website. Behind the scenes, the company stores everything in a small relational database.

We will create four tables:
| Table | What it stores |
|-------|---------------|
| `paper_products` | The catalog of papers (name, color, weight, price, stock) |
| `customers` | People who buy paper |
| `orders` | Purchase transactions linking customers to products |
| `refunds` | Refund requests on past orders |

Let's build this database from scratch.

In [5]:
#@title Create the Dunder Mifflin database { display-mode: "form" }
DB_PATH = "../dunder_mifflin.db"

# Remove old DB if it exists so we start fresh
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.executescript("""
    CREATE TABLE paper_products (
        id          INTEGER PRIMARY KEY,
        name        TEXT NOT NULL,
        color       TEXT,
        weight_gsm  INTEGER,
        price       REAL,
        stock_qty   INTEGER,
        category    TEXT
    );

    CREATE TABLE customers (
        id          INTEGER PRIMARY KEY,
        name        TEXT NOT NULL,
        email       TEXT UNIQUE,
        region      TEXT,
        signup_date DATE
    );

    CREATE TABLE orders (
        id          INTEGER PRIMARY KEY,
        customer_id INTEGER REFERENCES customers(id),
        product_id  INTEGER REFERENCES paper_products(id),
        quantity    INTEGER,
        order_date  DATE,
        status      TEXT
    );

    CREATE TABLE refunds (
        id          INTEGER PRIMARY KEY,
        order_id    INTEGER REFERENCES orders(id),
        reason      TEXT,
        amount      REAL,
        refund_date DATE
    );
""")

conn.commit()
print(f"Database created: {DB_PATH}")

Database created: dunder_mifflin.db


In [6]:
#@title Populate with sample data { display-mode: "form" }
# Populate with sample data

cur.executescript("""
    INSERT INTO paper_products VALUES
        (1,  'Classic White A4',      'white',  80,  12.99, 500,  'copy'),
        (2,  'Premium White A4',      'white',  100, 18.50, 200,  'copy'),
        (3,  'Recycled Cream A4',     'cream',  90,  14.00, 350,  'copy'),
        (4,  'Bright Red A3',         'red',    120, 22.00, 75,   'specialty'),
        (5,  'Ocean Blue Cardstock',  'blue',   250, 35.00, 40,   'cardstock'),
        (6,  'Sunshine Yellow A4',    'yellow', 80,  13.50, 180,  'copy'),
        (7,  'Jet Black Cover',       'black',  300, 42.00, 25,   'cardstock'),
        (8,  'Pastel Pink Letter',    'pink',   75,  11.00, 420,  'copy'),
        (9,  'Green Envelopes',       'green',  120, 9.50,  600,  'envelope'),
        (10, 'Gold Foil Invitation',  'gold',   200, 55.00, 15,   'specialty');

    INSERT INTO customers VALUES
        (1, 'Michael Scott',    'michael@dundermifflin.com', 'Northeast', '2023-01-10'),
        (2, 'Jim Halpert',      'jim@dundermifflin.com',     'Northeast', '2023-02-15'),
        (3, 'Pam Beesly',       'pam@dundermifflin.com',     'Northeast', '2023-02-15'),
        (4, 'Dwight Schrute',   'dwight@dundermifflin.com',  'Midwest',   '2023-03-01'),
        (5, 'Angela Martin',    'angela@dundermifflin.com',  'Midwest',   '2023-04-20'),
        (6, 'Stanley Hudson',   'stanley@dundermifflin.com', 'Southeast', '2023-05-11'),
        (7, 'Kevin Malone',     'kevin@dundermifflin.com',   'Northeast', '2023-06-30'),
        (8, 'Oscar Martinez',   'oscar@dundermifflin.com',   'West',      '2024-01-05');

    INSERT INTO orders VALUES
        (1,  1, 1,  50,  '2024-01-15', 'delivered'),
        (2,  1, 4,  10,  '2024-01-20', 'delivered'),
        (3,  2, 1,  100, '2024-02-01', 'delivered'),
        (4,  3, 8,  30,  '2024-02-14', 'delivered'),
        (5,  4, 7,  5,   '2024-03-05', 'delivered'),
        (6,  4, 10, 2,   '2024-03-05', 'shipped'),
        (7,  5, 3,  200, '2024-03-18', 'delivered'),
        (8,  6, 6,  25,  '2024-04-01', 'pending'),
        (9,  7, 5,  8,   '2024-04-10', 'shipped'),
        (10, 2, 9,  300, '2024-04-22', 'delivered'),
        (11, 8, 2,  60,  '2024-05-03', 'delivered'),
        (12, 1, 10, 3,   '2024-05-10', 'pending');

    INSERT INTO refunds VALUES
        (1, 2,  'wrong color received',     22.00,  '2024-02-01'),
        (2, 5,  'damaged during shipping',  42.00,  '2024-03-20'),
        (3, 7,  'order too large',          700.00, '2024-04-02');
""")

conn.commit()
print("Sample data inserted.")

Sample data inserted.


### Let's look at the data

We display each table so you can see what Dunder Mifflin's database looks like.

In [7]:
#@title Display database tables { display-mode: "form" }
from IPython.display import HTML, display

TABLE_COLORS = {
    "paper_products": "#6366f1",
    "customers": "#10b981",
    "orders": "#f59e0b",
    "refunds": "#ef4444",
}

def render_table_html(table_name, df, color):
    header = ''.join(
        f'<th style="padding:8px 14px;text-align:left;color:white;background:{color};">{col}</th>'
        for col in df.columns
    )
    rows = ''
    for i, (_, row) in enumerate(df.iterrows()):
        bg = '#f8fafc' if i % 2 else 'white'
        cells_html = ''.join(
            f'<td style="padding:7px 14px;border-bottom:1px solid #e2e8f0;font-size:13px;">{v}</td>'
            for v in row
        )
        rows += f'<tr style="background:{bg};">{cells_html}</tr>'

    return f'''
    <div style="margin-bottom:24px;">
        <div style="background:{color};color:white;padding:10px 18px;border-radius:10px 10px 0 0;
                    font-weight:700;font-size:15px;letter-spacing:0.5px;">
            {table_name}
            <span style="font-weight:400;font-size:12px;opacity:0.8;margin-left:8px;">{len(df)} rows</span>
        </div>
        <div style="overflow-x:auto;border:1px solid #e2e8f0;border-top:none;border-radius:0 0 10px 10px;">
            <table style="width:100%;border-collapse:collapse;font-family:system-ui,sans-serif;">
                <thead><tr>{header}</tr></thead>
                <tbody>{rows}</tbody>
            </table>
        </div>
    </div>'''

html_parts = []
for table in ["paper_products", "customers", "orders", "refunds"]:
    df = pd.read_sql_query(f"SELECT * FROM {table}", conn)
    html_parts.append(render_table_html(table, df, TABLE_COLORS[table]))

display(HTML(
    '<div style="max-width:900px;">'
    + ''.join(html_parts)
    + '</div>'
))



  TABLE: paper_products


,id,name,color,weight_gsm,price,stock_qty,category
0,1,Classic White A4,white,80,12.99,500,copy
1,2,Premium White A4,white,100,18.50,200,copy
2,3,Recycled Cream A4,cream,90,14.00,350,copy
3,4,Bright Red A3,red,120,22.00,75,specialty
4,5,Ocean Blue Cardstock,blue,250,35.00,40,cardstock
5,6,Sunshine Yellow A4,yellow,80,13.50,180,copy
6,7,Jet Black Cover,black,300,42.00,25,cardstock
7,8,Pastel Pink Letter,pink,75,11.00,420,copy
8,9,Green Envelopes,green,120,9.50,600,envelope
9,10,Gold Foil Invitation,gold,200,55.00,15,specialty



  TABLE: customers


,id,name,email,region,signup_date
0,1,Michael Scott,michael@dundermifflin.com,Northeast,2023-01-10
1,2,Jim Halpert,jim@dundermifflin.com,Northeast,2023-02-15
2,3,Pam Beesly,pam@dundermifflin.com,Northeast,2023-02-15
3,4,Dwight Schrute,dwight@dundermifflin.com,Midwest,2023-03-01
4,5,Angela Martin,angela@dundermifflin.com,Midwest,2023-04-20
5,6,Stanley Hudson,stanley@dundermifflin.com,Southeast,2023-05-11
6,7,Kevin Malone,kevin@dundermifflin.com,Northeast,2023-06-30
7,8,Oscar Martinez,oscar@dundermifflin.com,West,2024-01-05



  TABLE: orders


,id,customer_id,product_id,quantity,order_date,status
0,1,1,1,50,2024-01-15,delivered
1,2,1,4,10,2024-01-20,delivered
2,3,2,1,100,2024-02-01,delivered
3,4,3,8,30,2024-02-14,delivered
4,5,4,7,5,2024-03-05,delivered
5,6,4,10,2,2024-03-05,shipped
6,7,5,3,200,2024-03-18,delivered
7,8,6,6,25,2024-04-01,pending
8,9,7,5,8,2024-04-10,shipped
9,10,2,9,300,2024-04-22,delivered



  TABLE: refunds


,id,order_id,reason,amount,refund_date
0,1,2,wrong color received,22.0,2024-02-01
1,2,5,damaged during shipping,42.0,2024-03-20
2,3,7,order too large,700.0,2024-04-02


### What the LLM sees: raw schema

When we ask an LLM to write SQL, all it gets is the **schema** — table names and column names with their types. Let's see how that looks.

In [8]:
#@title Show raw schema { display-mode: "form" }
# Show raw schema — this is what a naive Text-to-SQL system would give the LLM
for table in ["paper_products", "customers", "orders", "refunds"]:
    print(f"\n-- {table} --")
    rows = cur.execute(f"PRAGMA table_info({table})").fetchall()
    for row in rows:
        col_id, name, dtype, notnull, default, pk = row
        flags = []
        if pk:
            flags.append("PK")
        if notnull:
            flags.append("NOT NULL")
        print(f"  {name:20s} {dtype:10s} {' '.join(flags)}")

print("\n--- That's it. No descriptions, no examples, no relationships explained. ---")
print("Can the LLM figure out what 'weight_gsm' means? Or that 'status' is one of {delivered, shipped, pending}?")


-- paper_products --
  id                   INTEGER    PK
  name                 TEXT       NOT NULL
  color                TEXT       
  weight_gsm           INTEGER    
  price                REAL       
  stock_qty            INTEGER    
  category             TEXT       

-- customers --
  id                   INTEGER    PK
  name                 TEXT       NOT NULL
  email                TEXT       
  region               TEXT       
  signup_date          DATE       

-- orders --
  id                   INTEGER    PK
  customer_id          INTEGER    
  product_id           INTEGER    
  quantity             INTEGER    
  order_date           DATE       
  status               TEXT       

-- refunds --
  id                   INTEGER    PK
  order_id             INTEGER    
  reason               TEXT       
  amount               REAL       
  refund_date          DATE       

--- That's it. No descriptions, no examples, no relationships explained. ---
Can the LLM figure out wh

---

## Section 2 — M-Schema Generation with XiYan-DBDescGen

The raw schema is ambiguous. Column names like `weight_gsm`, `status`, or `region` don't tell the LLM much. **M-Schema** fixes this by enriching each field with:

- **Category** — is this field a Code (ID), Enum (fixed values), Measure (numeric metric), DateTime, or Text?
- **Dimension vs. Measure** — is it something you group by (dimension) or aggregate (measure)?
- **Natural language descriptions** — auto-generated by the LLM itself
- **Example values** — so the LLM knows what valid data looks like

The XiYan pipeline has two main steps:
1. `fields_category()` — classify every column
2. `table_and_column_desc_generation()` — generate descriptions for tables and columns

### Step 2.1 — Initialize the Schema Engine

We connect to our Dunder Mifflin database and create a `SchemaEngine`. Setting `comment_mode='generation'` tells XiYan to generate all descriptions from scratch (our SQLite DB has no comments).

In [9]:
#@title Initialize SchemaEngine { display-mode: "form" }
from schema_engine import SchemaEngine

# Close the raw sqlite3 connection — SchemaEngine uses SQLAlchemy
conn.close()

# Create a SQLAlchemy engine pointing to our DB
db_engine = create_engine(f"sqlite:///{os.path.abspath(DB_PATH)}")

# Initialize the SchemaEngine
schema_engine = SchemaEngine(
    db_engine,
    llm=llm,
    db_name="dunder_mifflin",
    comment_mode="generation",  # generate all descriptions from scratch
)

print("SchemaEngine initialized.")
print(f"Tables found: {list(schema_engine.mschema.tables.keys())}")

SchemaEngine initialized.
Tables found: ['orders', 'paper_products', 'customers', 'refunds']


### Step 2.2 — Classify Fields

The first pass asks the LLM to classify every column. For each field it determines:
- **Category**: Code, Enum, Measure, DateTime, or Text
- **Dimension vs Measure**: is this something you group by or aggregate?

Watch the output — notice how `status` gets classified as **Enum**, `price` as **Measure**, and `id` as **Code**.

In [10]:
#@title Classify fields (LLM calls) { display-mode: "form" }
# This calls the LLM for each field — it may take a minute or two
schema_engine.fields_category()

Table Name:  orders
Field Name:  id
[Field Info]
Field Name: id
Field Type: INTEGER
Primary Key (or part of composite PK): True
UNIQUE: True
NULLABLE: True
COUNT: 12
COUNT(DISTINCT): 12
MAX: 12
MIN: 1
AVG: 6.5
MAX(CHAR_LENGTH): 2
MIN(CHAR_LENGTH): 1
Value Examples: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
{'category': 'Code', 'dim_or_meas': 'Dimension'}
Field Name:  customer_id
[Field Info]
Field Name: customer_id
Field Type: INTEGER
Primary Key (or part of composite PK): False
UNIQUE: False
NULLABLE: True
COUNT: 12
COUNT(DISTINCT): 8
MAX: 8
MIN: 1
AVG: 3.6666666666666665
MAX(CHAR_LENGTH): 1
MIN(CHAR_LENGTH): 1
Value Examples: [1, 2, 3, 4, 5, 6, 7, 8]
{'category': 'Code', 'dim_or_meas': 'Dimension'}
Field Name:  product_id
[Field Info]
Field Name: product_id
Field Type: INTEGER
Primary Key (or part of composite PK): False
UNIQUE: False
NULLABLE: True
COUNT: 12
COUNT(DISTINCT): 10
MAX: 10
MIN: 1
AVG: 5.5
MAX(CHAR_LENGTH): 2
MIN(CHAR_LENGTH): 1
Value Examples: [1, 4, 8, 7, 10, 3, 6, 5, 9, 2]
{'ca

In [ ]:
#@title Visualization: Field Classification Dashboard { display-mode: "form" }
# Let's inspect the classification results in a readable table
rows = []
for table_name, table_info in schema_engine.mschema.tables.items():
    for field_name, field_info in table_info["fields"].items():
        rows.append({
            "table": table_name,
            "column": field_name,
            "type": field_info.get("type", ""),
            "category": field_info.get("category", ""),
            "dim_or_meas": field_info.get("dim_or_meas", ""),
            "examples": field_info.get("examples", [])[:3],
        })

df_fields = pd.DataFrame(rows)

from IPython.display import HTML, display

# ── Visual: Field Classification Dashboard ──

CATEGORY_COLORS = {
    "Code": "#6366f1",       # indigo
    "Enum": "#f59e0b",       # amber
    "Measure": "#10b981",    # emerald
    "DateTime": "#3b82f6",   # blue
    "Text": "#ef4444",       # red
}
DIM_MEAS_ICONS = {
    "Dimension": "&#9642;",  # small square
    "Measure": "&#9650;",    # triangle
}

def render_classification_html(df):
    tables = df["table"].unique()
    cards_html = ""

    for tbl in tables:
        tdf = df[df["table"] == tbl]
        fields_html = ""
        for _, row in tdf.iterrows():
            cat = row["category"]
            color = CATEGORY_COLORS.get(cat, "#94a3b8")
            dm = row["dim_or_meas"]
            dm_icon = DIM_MEAS_ICONS.get(dm, "")
            examples = row["examples"]
            ex_str = ", ".join(str(e) for e in examples) if examples else ""
            ex_html = f'<span style="color:#64748b;font-size:11px;">e.g. {ex_str}</span>' if ex_str else ""

            fields_html += f"""
            <div style="display:flex;align-items:center;gap:8px;padding:6px 0;border-bottom:1px solid #f1f5f9;">
                <span style="font-family:monospace;font-weight:600;min-width:140px;color:#1e293b;">{row["column"]}</span>
                <span style="font-size:11px;color:#94a3b8;min-width:70px;">{row["type"]}</span>
                <span style="background:{color};color:white;padding:2px 10px;border-radius:12px;font-size:12px;font-weight:500;">{cat}</span>
                <span style="font-size:12px;color:#475569;">{dm_icon} {dm}</span>
                {ex_html}
            </div>"""

        cards_html += f"""
        <div style="background:white;border:1px solid #e2e8f0;border-radius:12px;padding:20px;margin-bottom:16px;box-shadow:0 1px 3px rgba(0,0,0,0.06);">
            <div style="font-size:16px;font-weight:700;color:#0f172a;margin-bottom:12px;padding-bottom:8px;border-bottom:2px solid #e2e8f0;">
                &#128202; {tbl}
            </div>
            {fields_html}
        </div>"""

    # Legend
    legend = " ".join(
        f'<span style="display:inline-flex;align-items:center;gap:4px;margin-right:14px;">'
        f'<span style="background:{c};width:12px;height:12px;border-radius:50%;display:inline-block;"></span>'
        f'<span style="font-size:12px;color:#475569;">{cat}</span></span>'
        for cat, c in CATEGORY_COLORS.items()
    )

    html = f"""
    <div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;max-width:800px;">
        <div style="font-size:20px;font-weight:700;color:#0f172a;margin-bottom:4px;">Field Classifications</div>
        <div style="color:#64748b;font-size:13px;margin-bottom:12px;">Each column classified by the LLM into a category and dimension/measure role</div>
        <div style="margin-bottom:16px;padding:8px 12px;background:#f8fafc;border-radius:8px;">{legend}</div>
        {cards_html}
    </div>"""
    return html

display(HTML(render_classification_html(df_fields)))

### Step 2.3 — Generate Descriptions

Now the LLM generates natural language descriptions for every table and column. This is the "fine-to-coarse" step — it first understands the overall database, then describes each field in context.

In [13]:
#@title Generate descriptions (LLM calls) { display-mode: "form" }
# Generate table and column descriptions — this also takes a minute or two
schema_engine.table_and_column_desc_generation(language='EN')

DB INFO:  The database schema `dunder_mifflin` covers the domain of a paper products business, specifically focusing on order management, product inventory, customer data, and refunds. It stores detailed information about orders placed by customers, including the specific products ordered, quantities, order dates, and statuses. The `paper_products` table contains details about various paper items such as their names, colors, weights, prices, stock quantities, and categories. The `customers` table holds customer-related data like names, email addresses, regions of operation, and signup dates. Finally, the `refunds` table tracks refunds associated with specific orders, recording reasons for refunds and amounts refunded along with the dates when these refunds were processed. Overall, this database provides a comprehensive view of sales, inventory management, customer interactions, and customer service activities related to paper products.
In the context of a paper products business using 

In [14]:
#@title Visualization: Generated Descriptions { display-mode: "form" }
# ── Visual: Generated Descriptions ──

def render_descriptions_html(mschema):
    cards_html = ""

    for table_name, table_info in mschema.tables.items():
        table_desc = table_info.get("comment", "")
        table_desc_html = f'<div style="color:#475569;font-size:13px;font-style:italic;margin-bottom:14px;padding:8px 12px;background:#f0fdf4;border-left:3px solid #10b981;border-radius:4px;">{table_desc}</div>' if table_desc else ""

        fields_html = ""
        for field_name, field_info in table_info["fields"].items():
            desc = field_info.get("comment", "")
            cat = field_info.get("category", "")
            color = CATEGORY_COLORS.get(cat, "#94a3b8")

            desc_span = f'<span style="color:#475569;font-size:13px;">{desc}</span>' if desc else '<span style="color:#cbd5e1;font-size:12px;font-style:italic;">no description</span>'

            fields_html += f"""
            <div style="display:flex;align-items:baseline;gap:10px;padding:5px 0;border-bottom:1px solid #f1f5f9;">
                <span style="font-family:monospace;font-weight:600;min-width:140px;color:#1e293b;flex-shrink:0;">{field_name}</span>
                <span style="background:{color};color:white;padding:1px 8px;border-radius:10px;font-size:11px;flex-shrink:0;">{cat}</span>
                <span style="color:#94a3b8;flex-shrink:0;">&#8594;</span>
                {desc_span}
            </div>"""

        cards_html += f"""
        <div style="background:white;border:1px solid #e2e8f0;border-radius:12px;padding:20px;margin-bottom:16px;box-shadow:0 1px 3px rgba(0,0,0,0.06);">
            <div style="font-size:16px;font-weight:700;color:#0f172a;margin-bottom:8px;">&#128221; {table_name}</div>
            {table_desc_html}
            {fields_html}
        </div>"""

    html = f"""
    <div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;max-width:850px;">
        <div style="font-size:20px;font-weight:700;color:#0f172a;margin-bottom:4px;">Generated Descriptions</div>
        <div style="color:#64748b;font-size:13px;margin-bottom:16px;">The LLM auto-generated these descriptions by analyzing table structure, sample data, and field relationships</div>
        {cards_html}
    </div>"""
    return html

display(HTML(render_descriptions_html(schema_engine.mschema)))

### Step 2.4 — The Final M-Schema

This is what the LLM will actually receive when generating SQL. Compare it to the raw `PRAGMA table_info` output from Section 1 — much richer!

In [15]:
#@title Visualization: M-Schema output { display-mode: "form" }
# ── Visual: M-Schema (the enriched schema the LLM receives) ──

import html as html_mod

mschema_str = schema_engine.mschema.to_mschema()

def render_mschema_html(mschema_text):
    """Render M-Schema text as a styled, syntax-highlighted block."""
    lines = mschema_text.split("\n")
    styled_lines = []

    for line in lines:
        escaped = html_mod.escape(line)
        if escaped.startswith("【") or escaped.startswith("["):
            # Section headers
            styled_lines.append(f'<div style="color:#6366f1;font-weight:700;font-size:14px;margin-top:16px;margin-bottom:4px;">{escaped}</div>')
        elif escaped.startswith("# Table:"):
            # Table headers
            parts = escaped.split(",", 1)
            table_part = parts[0]
            desc_part = f'<span style="color:#10b981;font-weight:400;font-style:italic;">, {parts[1]}</span>' if len(parts) > 1 else ""
            styled_lines.append(f'<div style="color:#0f172a;font-weight:700;font-size:15px;margin-top:14px;border-bottom:1px solid #e2e8f0;padding-bottom:4px;">{table_part}{desc_part}</div>')
        elif escaped.strip().startswith("(") and ":" in escaped:
            # Field lines — highlight field name, type, and extras
            indent = len(escaped) - len(escaped.lstrip())
            content = escaped.strip()
            # Color the field name
            if content.startswith("("):
                inner = content[1:]  # strip leading (
                if ":" in inner:
                    fname, rest = inner.split(":", 1)
                    # Split rest into type and description parts
                    rest_parts = rest.split(",", 1)
                    ftype = rest_parts[0].strip()
                    extras = f',<span style="color:#475569;">{rest_parts[1]}</span>' if len(rest_parts) > 1 else ""
                    content = (f'(<span style="color:#6366f1;font-weight:600;">{fname}</span>'
                              f':<span style="color:#f59e0b;">{ftype}</span>{extras}')
            styled_lines.append(f'<div style="font-family:monospace;font-size:13px;padding:2px 0 2px {indent*8+16}px;color:#334155;">{content}</div>')
        elif "=" in escaped and not escaped.startswith("[") and not escaped.startswith("【"):
            # Foreign key lines
            parts = escaped.split("=")
            if len(parts) == 2:
                styled_lines.append(f'<div style="font-family:monospace;font-size:13px;padding:2px 0 2px 16px;color:#64748b;"><span style="color:#3b82f6;">{parts[0]}</span> = <span style="color:#3b82f6;">{parts[1]}</span></div>')
            else:
                styled_lines.append(f'<div style="font-family:monospace;font-size:13px;padding:2px 0 2px 16px;color:#64748b;">{escaped}</div>')
        elif escaped.strip() in ("[", "]"):
            styled_lines.append(f'<div style="font-family:monospace;font-size:13px;color:#94a3b8;padding-left:8px;">{escaped}</div>')
        else:
            styled_lines.append(f'<div style="font-family:monospace;font-size:13px;color:#334155;padding:1px 0;">{escaped}</div>')

    body = "\n".join(styled_lines)
    return f"""
    <div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;max-width:900px;">
        <div style="font-size:20px;font-weight:700;color:#0f172a;margin-bottom:4px;">M-Schema Output</div>
        <div style="color:#64748b;font-size:13px;margin-bottom:12px;">This is the enriched schema that gets sent to the LLM for SQL generation — compare it with the raw PRAGMA output from Section 1</div>
        <div style="background:#fafbfc;border:1px solid #e2e8f0;border-radius:12px;padding:20px;box-shadow:0 1px 3px rgba(0,0,0,0.06);overflow-x:auto;">
            {body}
        </div>
    </div>"""

display(HTML(render_mschema_html(mschema_str)))

In [16]:
#@title Save schema to JSON { display-mode: "form" }
# Save the enriched schema to JSON for later reuse
schema_engine.mschema.save("dunder_mifflin_schema.json")
print("Schema saved to dunder_mifflin_schema.json")

Schema saved to dunder_mifflin_schema.json


### What just happened?

Let's recap the XiYan M-Schema pipeline:

| Step | Method | What it does |
|------|--------|-------------|
| Init | `SchemaEngine(...)` | Reads the database schema, collects sample values for each column |
| 1 | `fields_category()` | LLM classifies each field as Code/Enum/Measure/DateTime/Text |
| 2 | `table_and_column_desc_generation()` | LLM generates natural language descriptions for tables and columns |
| Output | `mschema.to_mschema()` | Produces a compact, enriched schema string ready for SQL generation |

**Key insight**: We used an LLM to *understand* the database so that a (potentially different) LLM can later *query* it more accurately. The schema became a better "tool manual".

---

*Next: Section 3 — We'll use this enriched schema to generate SQL from natural language questions.*

---

## Section 3 — Schema Filtering: Helping the LLM Focus

With a large database, sending the **entire** enriched schema to the LLM is wasteful and confusing. Imagine a database with 50 tables — most of them are irrelevant to the user's question.

**XiYan-SQL's first stage** is a schema filter: given a specific question, narrow down which tables and columns the LLM actually needs to see.

The idea:
1. Extract **keywords** from the question (using the LLM)
2. **Match** those keywords against table names, column names, and sample values
3. Build a **filtered M-Schema** containing only the relevant subset

This is our first example of **orchestration** — using one LLM call to prepare the context for a later LLM call.

### Step 3.1 — Define a Question

Let's pick a natural language question that a Dunder Mifflin employee might ask.

In [17]:
#@title Define the question { display-mode: "form" }
QUESTION = "How many red papers are currently in stock?"

print(f"Question: {QUESTION}")

Question: How many red papers are currently in stock?


### Step 3.2 — Extract Keywords with the LLM

First, we ask the LLM to pull out the **key terms** from the question — the words that likely correspond to table names, column names, or data values in the database.

In [18]:
import json as json_mod
import re
from IPython.display import HTML, display

KEYWORD_EXTRACTION_PROMPT = PromptTemplate(
    """You are a data analyst. Given a natural language question about a database, extract the key terms that are likely to match table names, column names, or data values.

Return a JSON list of lowercase keywords. Only include meaningful terms — skip filler words like "how", "many", "currently", "the", "are", "is", "what", "which".

Question: {question}

```json
["keyword1", "keyword2", ...]
```
""", prompt_type=PromptType.CUSTOM)

# Ask the LLM to extract keywords
raw_response = llm.predict(KEYWORD_EXTRACTION_PROMPT, question=QUESTION)

# Parse the keywords from the response
try:
    json_match = re.search(r'```json\s*(.*?)\s*```', raw_response, re.DOTALL)
    if json_match:
        keywords = json_mod.loads(json_match.group(1))
    else:
        keywords = json_mod.loads(raw_response.strip())
except Exception:
    keywords = [w.strip().lower().strip('"[]') for w in raw_response.split(',')]

keywords = [k.lower().strip() for k in keywords if k.strip()]

# Render nicely
kw_pills = ' '.join(
    f'<span style="background:#eef2ff;color:#4338ca;padding:5px 14px;border-radius:20px;'
    f'font-size:13px;font-weight:600;border:1px solid #c7d2fe;">{kw}</span>'
    for kw in keywords
)

display(HTML(f'''
<div style="font-family:system-ui,sans-serif;max-width:700px;">
  <div style="background:white;border:1px solid #e2e8f0;border-radius:14px;overflow:hidden;">
    <div style="background:linear-gradient(135deg,#6366f1,#8b5cf6);color:white;padding:14px 20px;">
      <div style="font-size:14px;font-weight:700;">Keyword Extraction</div>
      <div style="font-size:12px;opacity:0.8;margin-top:2px;">Question &rarr; Key terms for schema matching</div>
    </div>
    <div style="padding:16px 20px;">
      <div style="font-size:12px;color:#64748b;margin-bottom:6px;font-weight:600;">QUESTION</div>
      <div style="font-size:14px;color:#0f172a;padding:10px 14px;background:#f8fafc;border-radius:8px;
                  border-left:3px solid #6366f1;margin-bottom:16px;font-style:italic;">
        &ldquo;{QUESTION}&rdquo;
      </div>
      <div style="font-size:12px;color:#64748b;margin-bottom:8px;font-weight:600;">EXTRACTED KEYWORDS</div>
      <div style="display:flex;flex-wrap:wrap;gap:8px;">{kw_pills}</div>
    </div>
  </div>
</div>
'''))


LLM response: ```json
["red", "papers", "stock"]
```

Extracted keywords: ['red', 'papers', 'stock']


### Step 3.3 — Match Keywords Against the Schema

Now we match the extracted keywords against our enriched M-Schema — checking table names, column names, column descriptions, and sample values. Any column that matches gets selected.

In [19]:
#@title Keyword matching function { display-mode: "form" }
def match_keywords_to_schema(mschema, keywords):
    """
    Simple keyword matching against the M-Schema.
    For each keyword, check if it appears in:
      - table names
      - column names
      - column descriptions
      - sample values
    Returns a list of matched columns as "table_name.column_name".
    """
    matched_columns = set()

    for table_name, table_info in mschema.tables.items():
        table_matched = False

        # Check if any keyword matches the table name
        for kw in keywords:
            if kw in table_name.lower():
                table_matched = True
                break

        for field_name, field_info in table_info["fields"].items():
            column_matched = False
            match_reasons = []

            for kw in keywords:
                # Match against column name
                if kw in field_name.lower():
                    column_matched = True
                    match_reasons.append(f"column name contains '{kw}'")

                # Match against column description
                desc = field_info.get("comment", "").lower()
                if desc and kw in desc:
                    column_matched = True
                    match_reasons.append(f"description contains '{kw}'")

                # Match against sample values
                examples = field_info.get("examples", [])
                for ex in examples:
                    if kw in str(ex).lower():
                        column_matched = True
                        match_reasons.append(f"value '{ex}' matches '{kw}'")
                        break

            if column_matched or table_matched:
                col_ref = f"{table_name}.{field_name}"
                matched_columns.add(col_ref)
                if match_reasons:
                    print(f"  MATCH: {col_ref:35s} <- {', '.join(match_reasons)}")

    return sorted(matched_columns)


print(f"Keywords: {keywords}\n")
selected_columns = match_keywords_to_schema(schema_engine.mschema, keywords)
print(f"\n{len(selected_columns)} columns selected out of "
      f"{sum(len(t['fields']) for t in schema_engine.mschema.tables.values())} total.")

Keywords: ['red', 'papers', 'stock']

  MATCH: orders.quantity                     <- description contains 'red'
  MATCH: orders.status                       <- description contains 'red', value 'delivered' matches 'red'
  MATCH: paper_products.name                 <- value 'Bright Red A3' matches 'red', value 'Ocean Blue Cardstock' matches 'stock'
  MATCH: paper_products.color                <- value 'red' matches 'red'
  MATCH: paper_products.stock_qty            <- column name contains 'stock', description contains 'stock'
  MATCH: paper_products.category             <- value 'cardstock' matches 'stock'

6 columns selected out of 23 total.


### Step 3.4 — Build the Filtered M-Schema

Now we use `to_mschema(selected_columns=[...])` to produce a compact schema that only includes the matched columns. Let's compare it side-by-side with the full schema.

In [20]:
#@title Generate filtered M-Schema { display-mode: "form" }
# Generate the filtered M-Schema
filtered_mschema_str = schema_engine.mschema.to_mschema(selected_columns=selected_columns)

# Also get the full one for comparison
full_mschema_str = schema_engine.mschema.to_mschema()

print("=== FILTERED M-Schema ===")
print(filtered_mschema_str)
print(f"\n--- Full schema: {len(full_mschema_str)} chars | Filtered: {len(filtered_mschema_str)} chars ---")
print(f"--- Reduction: {100 - len(filtered_mschema_str) * 100 // len(full_mschema_str)}% fewer tokens sent to the LLM ---")

=== FILTERED M-Schema ===
【DB_ID】 dunder_mifflin
【Schema】
# Table: orders, The orders table stores order-related metrics in a time dimension, including order date (order_date), and other dimensions such as customer ID (customer_id) and product ID (product_id). It tracks the quantity of products ordered (quantity) and their status (status).
[
(quantity:INTEGER, Quantity of ordered products., Examples: [50, 10, 100]),
(status:TEXT, Order status: delivered, shipped, pending., Examples: [delivered, shipped, pending])
]
# Table: paper_products, The paper_products table stores product dimension data including name, color, weight, price, stock quantity, and category, but no explicit time dimension is present.
[
(name:TEXT, The specific name of each paper product., Examples: [Classic White A4, Premium White A4, Recycled Cream A4]),
(color:TEXT, Product color in text format., Examples: [white, cream, red]),
(stock_qty:INTEGER, Quantity in stock for each product., Examples: [500, 200, 350]),
(ca

In [21]:
#@title Visualization: Full vs Filtered M-Schema { display-mode: "form" }
# ── Visual: Full vs Filtered M-Schema side-by-side ──
import html as html_mod

def render_schema_comparison_html(full_text, filtered_text, question, keywords, selected_cols, total_cols):
    """Render a side-by-side comparison of full vs filtered M-Schema."""

    def schema_to_html_block(text):
        lines = text.split("\n")
        parts = []
        for line in lines:
            escaped = html_mod.escape(line)
            if escaped.startswith("【") or escaped.startswith("["):
                parts.append(f'<div style="color:#6366f1;font-weight:700;font-size:13px;margin-top:10px;">{escaped}</div>')
            elif escaped.startswith("# Table:"):
                parts.append(f'<div style="color:#0f172a;font-weight:700;font-size:14px;margin-top:10px;border-bottom:1px solid #e2e8f0;padding-bottom:3px;">{escaped}</div>')
            elif escaped.strip().startswith("(") and ":" in escaped:
                parts.append(f'<div style="font-family:monospace;font-size:12px;padding:1px 0 1px 12px;color:#334155;">{escaped.strip()}</div>')
            elif escaped.strip() in ("[", "]"):
                parts.append(f'<div style="font-family:monospace;font-size:12px;color:#94a3b8;">{escaped}</div>')
            else:
                parts.append(f'<div style="font-family:monospace;font-size:12px;color:#334155;">{escaped}</div>')
        return "\n".join(parts)

    full_html = schema_to_html_block(full_text)
    filtered_html = schema_to_html_block(filtered_text)

    reduction = 100 - len(filtered_text) * 100 // len(full_text) if len(full_text) > 0 else 0

    kw_pills = " ".join(
        f'<span style="background:#6366f1;color:white;padding:2px 10px;border-radius:12px;font-size:12px;font-weight:500;margin-right:4px;">{k}</span>'
        for k in keywords
    )

    html = f"""
    <div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;max-width:1200px;">
        <div style="font-size:20px;font-weight:700;color:#0f172a;margin-bottom:4px;">Schema Filtering Result</div>
        <div style="color:#64748b;font-size:13px;margin-bottom:12px;">
            Question: <span style="color:#0f172a;font-weight:600;">"{question}"</span>
        </div>
        <div style="margin-bottom:12px;">
            <span style="color:#64748b;font-size:13px;">Extracted keywords: </span>{kw_pills}
        </div>

        <!-- Stats bar -->
        <div style="display:flex;gap:16px;margin-bottom:16px;">
            <div style="flex:1;background:#f8fafc;border:1px solid #e2e8f0;border-radius:8px;padding:12px;text-align:center;">
                <div style="font-size:24px;font-weight:700;color:#6366f1;">{total_cols}</div>
                <div style="font-size:12px;color:#64748b;">Total columns</div>
            </div>
            <div style="flex:1;background:#f0fdf4;border:1px solid #bbf7d0;border-radius:8px;padding:12px;text-align:center;">
                <div style="font-size:24px;font-weight:700;color:#10b981;">{len(selected_cols)}</div>
                <div style="font-size:12px;color:#64748b;">Selected columns</div>
            </div>
            <div style="flex:1;background:#fefce8;border:1px solid #fde68a;border-radius:8px;padding:12px;text-align:center;">
                <div style="font-size:24px;font-weight:700;color:#f59e0b;">{reduction}%</div>
                <div style="font-size:12px;color:#64748b;">Reduction</div>
            </div>
        </div>

        <!-- Side by side -->
        <div style="display:flex;gap:16px;">
            <div style="flex:1;background:white;border:1px solid #e2e8f0;border-radius:12px;padding:16px;box-shadow:0 1px 3px rgba(0,0,0,0.06);overflow-x:auto;max-height:500px;overflow-y:auto;">
                <div style="font-size:14px;font-weight:700;color:#ef4444;margin-bottom:8px;">&#10060; Full Schema ({len(full_text)} chars)</div>
                {full_html}
            </div>
            <div style="flex:1;background:white;border:1px solid #bbf7d0;border-radius:12px;padding:16px;box-shadow:0 1px 3px rgba(0,0,0,0.06);overflow-x:auto;max-height:500px;overflow-y:auto;">
                <div style="font-size:14px;font-weight:700;color:#10b981;margin-bottom:8px;">&#9989; Filtered Schema ({len(filtered_text)} chars)</div>
                {filtered_html}
            </div>
        </div>
    </div>"""
    return html

total_col_count = sum(len(t['fields']) for t in schema_engine.mschema.tables.values())

display(HTML(render_schema_comparison_html(
    full_mschema_str,
    filtered_mschema_str,
    QUESTION,
    keywords,
    selected_columns,
    total_col_count
)))

### What just happened?

We implemented a simplified version of **XiYan-SQL Stage 1** — schema filtering:

| Step | What it does | Who does the work |
|------|-------------|-------------------|
| Keyword extraction | Pull key terms from the question | LLM |
| Schema matching | Find columns related to those keywords | Simple code (string matching) |
| Filtered M-Schema | Build a compact schema with only the relevant parts | `to_mschema(selected_columns=...)` |

**Key insight**: This is the first example of **orchestration**. We used one LLM call (keyword extraction) to prepare better input for a future LLM call (SQL generation). The LLM is helping itself by narrowing down the problem space.

In the real XiYan-SQL paper, this stage is much more sophisticated — it uses embeddings, cosine similarity, and multiple iterative passes. But even our simple string matching achieves significant schema reduction.

---

*Next: Section 4 — We'll use the filtered schema to generate multiple SQL candidates with different strategies.*

---

## Section 4 — Multi-Candidate SQL Generation

Instead of asking the LLM once and hoping for the best, **XiYan-SQL generates multiple SQL candidates** using different strategies. More shots = higher chance one is correct.

We'll generate 3 candidates using different prompt strategies:

| Strategy | Idea |
|----------|------|
| **Direct** | Simple prompt — just ask for the SQL |
| **Chain-of-Thought** | Ask the LLM to reason step-by-step before writing SQL |
| **With Evidence** | Inject hints from the M-Schema (enum values, descriptions) |

All three use the **filtered schema** from Section 3, and the same base LLM.

### Step 4.1 — Define the Three Prompt Strategies

Each strategy wraps the same question and schema in a different prompt structure.

In [22]:
from utils import extract_sql_from_llm_response

# ── Strategy A: Direct ──
DIRECT_PROMPT = PromptTemplate(
    """You are a SQLite data analyst. Given the following database schema:

{db_schema}

Write a SQL query to answer this question: {question}

Wrap the SQL in ```sql and ```.
""", prompt_type=PromptType.CUSTOM)


# ── Strategy B: Chain-of-Thought ──
COT_PROMPT = PromptTemplate(
    """You are a SQLite data analyst. Given the following database schema:

{db_schema}

Question: {question}

Think step by step before writing the SQL:
1. Which tables are needed?
2. What columns should be selected?
3. What WHERE conditions or JOINs are needed?
4. Is any aggregation (COUNT, SUM, etc.) required?

After your reasoning, write the final SQL query. Wrap it in ```sql and ```.
""", prompt_type=PromptType.CUSTOM)


# ── Strategy C: With Evidence ──
# Auto-generate hints from the M-Schema: enum values + column descriptions
def build_evidence(mschema, selected_columns):
    """Extract enum values and descriptions from selected columns as hints."""
    hints = []
    for col_ref in selected_columns:
        table, col = col_ref.split(".")
        field_info = mschema.get_field_info(table, col)
        if not field_info:
            continue

        desc = field_info.get("comment", "")
        category = field_info.get("category", "")
        examples = field_info.get("examples", [])

        if category == "Enum" and examples:
            hints.append(f"- Column '{table}.{col}' has possible values: {', '.join(str(e) for e in examples)}")
        elif desc:
            hints.append(f"- Column '{table}.{col}': {desc}")

    return "\n".join(hints) if hints else "No additional hints available."

evidence = build_evidence(schema_engine.mschema, selected_columns)
print("Auto-generated evidence/hints:")
print(evidence)

EVIDENCE_PROMPT = PromptTemplate(
    """You are a SQLite data analyst. Given the following database schema:

{db_schema}

And the following hints about the data:
{evidence}

Write a SQL query to answer this question: {question}

Wrap the SQL in ```sql and ```.
""", prompt_type=PromptType.CUSTOM)


strategies = [
    ("Direct",          DIRECT_PROMPT,   {"db_schema": filtered_mschema_str, "question": QUESTION}),
    ("Chain-of-Thought", COT_PROMPT,     {"db_schema": filtered_mschema_str, "question": QUESTION}),
    ("With Evidence",   EVIDENCE_PROMPT, {"db_schema": filtered_mschema_str, "question": QUESTION, "evidence": evidence}),
]

print(f"\n3 strategies defined for question: \"{QUESTION}\"")

Auto-generated evidence/hints:
- Column 'orders.quantity': Quantity of ordered products.
- Column 'orders.status' has possible values: delivered, shipped, pending
- Column 'paper_products.category': Product category classification.
- Column 'paper_products.color' has possible values: white, cream, red, blue, yellow, black, pink, green, gold
- Column 'paper_products.name': The specific name of each paper product.
- Column 'paper_products.stock_qty': Quantity in stock for each product.

3 strategies defined for question: "How many red papers are currently in stock?"


### Step 4.2 — Generate and Execute All Candidates

We run each strategy, extract the SQL, execute it against the database, and collect the results.

In [23]:
#@title Multi-candidate SQL generation { display-mode: "form" }
from sqlalchemy import text
from IPython.display import HTML, display
import html as html_mod

def execute_sql(engine, sql_query):
    """Execute SQL and return results. Raises on error (unlike schema_engine.fetch)."""
    if not sql_query.strip().endswith(';'):
        sql_query += ';'
    with engine.begin() as conn:
        cursor = conn.execute(text(sql_query))
        return cursor.fetchall()

STRATEGY_STYLES = {
    "Direct":           {"color": "#3b82f6", "icon": "&#9889;",   "gradient": "linear-gradient(135deg,#3b82f6,#2563eb)"},
    "Chain-of-Thought": {"color": "#8b5cf6", "icon": "&#129504;", "gradient": "linear-gradient(135deg,#8b5cf6,#7c3aed)"},
    "With Evidence":    {"color": "#f59e0b", "icon": "&#128270;", "gradient": "linear-gradient(135deg,#f59e0b,#d97706)"},
}

candidates = []
cards_html = ""

# Live progress handle
progress = display(HTML('<div style="font-family:system-ui,sans-serif;color:#94a3b8;font-size:13px;">Generating candidates...</div>'), display_id=True)

for idx, (strategy_name, prompt_template, prompt_args) in enumerate(strategies):
    sty = STRATEGY_STYLES.get(strategy_name, {"color": "#64748b", "icon": "", "gradient": "#64748b"})

    # Update progress
    progress.update(HTML(
        f'<div style="font-family:system-ui,sans-serif;font-size:13px;color:#64748b;">'
        f'Running strategy <strong style="color:{sty["color"]};">{strategy_name}</strong> ({idx+1}/3)...</div>'
    ))

    # Generate
    raw_response = llm.predict(prompt_template, **prompt_args)
    sql = extract_sql_from_llm_response(raw_response)

    # Execute
    try:
        result = execute_sql(db_engine, sql)
        success = True
        error_msg = None
    except Exception as e:
        result = None
        success = False
        error_msg = str(e)

    candidates.append({
        "strategy": strategy_name,
        "sql": sql,
        "raw_response": raw_response,
        "result": result,
        "success": success,
        "error": error_msg,
    })

    # Build card HTML
    status_icon = "&#9989;" if success else "&#10060;"
    status_text = f"Result: <strong>{result}</strong>" if success else f"<span style='color:#ef4444;'>Failed: {html_mod.escape(str(error_msg))}</span>"

    # Trim reasoning for display
    reasoning = raw_response.strip()
    # Remove the SQL block from the reasoning display
    reasoning_clean = reasoning
    sql_block_start = reasoning_clean.find('```sql')
    if sql_block_start != -1:
        before = reasoning_clean[:sql_block_start].strip()
        reasoning_clean = before if before else '(direct SQL output)'
    if len(reasoning_clean) > 300:
        reasoning_clean = reasoning_clean[:300] + '...'

    cards_html += f'''
    <div style="background:white;border:1px solid #e2e8f0;border-radius:14px;margin-bottom:14px;
                overflow:hidden;box-shadow:0 1px 3px rgba(0,0,0,0.04);">
      <div style="background:{sty['gradient']};color:white;padding:12px 18px;
                  display:flex;align-items:center;gap:10px;">
        <span style="font-size:18px;">{sty['icon']}</span>
        <span style="font-weight:700;font-size:14px;">{strategy_name}</span>
        <span style="margin-left:auto;font-size:18px;">{status_icon}</span>
      </div>
      <div style="padding:14px 18px;">
        <div style="font-size:11px;color:#64748b;font-weight:600;margin-bottom:4px;">LLM REASONING</div>
        <div style="font-size:12px;color:#475569;padding:8px 12px;background:#f8fafc;border-radius:8px;
                    margin-bottom:12px;line-height:1.5;white-space:pre-wrap;">{html_mod.escape(reasoning_clean)}</div>
        <div style="font-size:11px;color:#64748b;font-weight:600;margin-bottom:4px;">GENERATED SQL</div>
        <div style="font-family:monospace;font-size:12px;color:#1e293b;padding:10px 14px;background:#f1f5f9;
                    border-radius:8px;border-left:3px solid {sty['color']};margin-bottom:12px;
                    white-space:pre-wrap;">{html_mod.escape(sql)}</div>
        <div style="font-size:13px;color:#0f172a;">{status_text}</div>
      </div>
    </div>'''

# Final display
ok = sum(c['success'] for c in candidates)
summary_color = '#10b981' if ok == len(candidates) else '#f59e0b' if ok > 0 else '#ef4444'

progress.update(HTML(f'''
<div style="font-family:system-ui,sans-serif;max-width:750px;">
  <div style="display:flex;align-items:center;gap:10px;margin-bottom:16px;">
    <div style="background:{summary_color};color:white;padding:6px 14px;border-radius:20px;
                font-size:13px;font-weight:700;">{ok}/{len(candidates)} succeeded</div>
    <div style="font-size:13px;color:#64748b;">Multi-candidate SQL generation complete</div>
  </div>
  {cards_html}
</div>
'''))



  Strategy: Direct

LLM reasoning/response:
```sql
SELECT SUM(stock_qty) AS total_stock_red_papers
FROM paper_products
WHERE color = 'red';
```

This SQL query sums up the `stock_qty` for all products that have the color 'red', giving you the total number of red papers currently in stock....

Extracted SQL:
SELECT SUM(stock_qty) AS total_stock_red_papers
FROM paper_products
WHERE color = 'red';

Result: [(75,)]

  Strategy: Chain-of-Thought

LLM reasoning/response:
To answer the question of how many red papers are currently in stock, let's break down the steps:

1. **Tables Needed**: We need to use the `paper_products` table since we are interested in information about paper products.

2. **Columns to Select**: We should select the `stock_qty` column which represents the quantity in stock for each product.

3. **WHERE Conditions or JOINs**: 
   - We need a condition to filter the rows where the color of the paper is 'red'. This can be done using the `color...

Extracted SQL:
SELECT SU

### Step 4.3 — Compare Candidates

In [24]:
#@title Visualization: Candidate Comparison Cards { display-mode: "form" }
# ── Visual: Candidate Comparison Cards ──

STRATEGY_COLORS = {
    "Direct": "#3b82f6",
    "Chain-of-Thought": "#8b5cf6",
    "With Evidence": "#f59e0b",
}

def render_candidates_html(candidates, question):
    cards = ""
    for c in candidates:
        color = STRATEGY_COLORS.get(c["strategy"], "#64748b")
        status_icon = "&#9989;" if c["success"] else "&#10060;"
        status_color = "#10b981" if c["success"] else "#ef4444"
        status_text = "Executed successfully" if c["success"] else f"Failed: {c['error']}"

        # Format SQL with syntax coloring
        sql_escaped = html_mod.escape(c["sql"])

        # Format result
        if c["success"] and c["result"] is not None:
            result_rows = c["result"]
            if len(result_rows) == 0:
                result_html = '<span style="color:#94a3b8;font-style:italic;">Empty result set</span>'
            else:
                rows_str = "<br>".join(html_mod.escape(str(r)) for r in result_rows[:10])
                if len(result_rows) > 10:
                    rows_str += f"<br><span style='color:#94a3b8;'>... and {len(result_rows)-10} more rows</span>"
                result_html = f'<div style="font-family:monospace;font-size:12px;color:#0f172a;">{rows_str}</div>'
        else:
            result_html = f'<span style="color:#ef4444;font-size:12px;">{html_mod.escape(str(c["error"]))}</span>'

        cards += f"""
        <div style="flex:1;min-width:250px;background:white;border:1px solid #e2e8f0;border-top:4px solid {color};border-radius:12px;padding:16px;box-shadow:0 1px 3px rgba(0,0,0,0.06);">
            <div style="font-size:15px;font-weight:700;color:{color};margin-bottom:10px;">{c["strategy"]}</div>

            <div style="font-size:11px;font-weight:600;color:#94a3b8;text-transform:uppercase;margin-bottom:4px;">Generated SQL</div>
            <div style="background:#f8fafc;border:1px solid #e2e8f0;border-radius:8px;padding:10px;margin-bottom:12px;overflow-x:auto;">
                <pre style="margin:0;font-size:12px;color:#1e293b;white-space:pre-wrap;">{sql_escaped}</pre>
            </div>

            <div style="font-size:11px;font-weight:600;color:#94a3b8;text-transform:uppercase;margin-bottom:4px;">Execution</div>
            <div style="display:flex;align-items:center;gap:6px;margin-bottom:8px;">
                <span style="font-size:16px;">{status_icon}</span>
                <span style="font-size:12px;color:{status_color};font-weight:500;">{status_text}</span>
            </div>

            <div style="font-size:11px;font-weight:600;color:#94a3b8;text-transform:uppercase;margin-bottom:4px;">Result</div>
            <div style="background:#f0fdf4;border:1px solid #bbf7d0;border-radius:8px;padding:10px;min-height:30px;">
                {result_html}
            </div>
        </div>"""

    # Check if all successful candidates agree
    successful = [c for c in candidates if c["success"]]
    if len(successful) > 1:
        results_set = set(str(c["result"]) for c in successful)
        if len(results_set) == 1:
            agreement = '<span style="color:#10b981;font-weight:600;">&#9989; All successful candidates agree on the same result!</span>'
        else:
            agreement = f'<span style="color:#f59e0b;font-weight:600;">&#9888; Candidates disagree — {len(results_set)} different results. Selection needed.</span>'
    elif len(successful) == 1:
        agreement = '<span style="color:#3b82f6;">Only 1 candidate succeeded.</span>'
    else:
        agreement = '<span style="color:#ef4444;font-weight:600;">&#10060; All candidates failed!</span>'

    html = f"""
    <div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;max-width:1200px;">
        <div style="font-size:20px;font-weight:700;color:#0f172a;margin-bottom:4px;">SQL Candidate Comparison</div>
        <div style="color:#64748b;font-size:13px;margin-bottom:6px;">
            Question: <span style="color:#0f172a;font-weight:600;">"{html_mod.escape(question)}"</span>
        </div>
        <div style="margin-bottom:16px;font-size:13px;">{agreement}</div>
        <div style="display:flex;gap:16px;flex-wrap:wrap;">
            {cards}
        </div>
    </div>"""
    return html

display(HTML(render_candidates_html(candidates, QUESTION)))

### What just happened?

We implemented a simplified version of **XiYan-SQL Stage 2** — multi-candidate generation:

| Strategy | How it works | Why it helps |
|----------|-------------|-------------|
| Direct | Minimal prompt, one shot | Fast baseline |
| Chain-of-Thought | Forces step-by-step reasoning | Catches logical mistakes |
| With Evidence | Injects enum values and descriptions as hints | Reduces ambiguity |

**Key insight**: Diversity in prompt strategies is an **engineering solution** to LLM unreliability. The real XiYan-SQL paper goes further — it uses 5 different fine-tuned models to maximize diversity.

But what if a candidate has a syntax error? Let's fix that next.


---

## Section 5 — Self-Correction: Fixing Broken SQL

LLMs sometimes produce SQL that **doesn't execute** — wrong column names, syntax errors, or incorrect joins.

Instead of throwing away a failed candidate, we can **feed the error back** to the LLM and ask it to fix the query. This is XiYan-SQL's **self-correction** step.

The idea is simple:
1. Try to execute the SQL
2. If it fails, send the LLM: the original question + the broken SQL + the error message
3. Ask it to produce a corrected query
4. Repeat up to N times


### Step 5.1 — The Correction Prompt

We give the LLM the original question, the broken SQL, and the database error message.


In [ ]:
CORRECTION_PROMPT = PromptTemplate(
    """You are a SQLite data analyst. You wrote a SQL query that failed.

Database schema:
{db_schema}

Original question: {question}

Your SQL:
```sql
{broken_sql}
```

Error message:
{error_message}

Fix the SQL query. Wrap the corrected SQL in ```sql and ```.
""", prompt_type=PromptType.CUSTOM)


def self_correct(llm, question, schema_str, broken_sql, error_msg, max_retries=2):
    """
    Try to fix a broken SQL query by feeding the error back to the LLM.
    Returns (corrected_sql, result, success, attempts_log).
    """
    attempts = []
    current_sql = broken_sql
    current_error = error_msg

    for attempt in range(1, max_retries + 1):
        print(f"  Correction attempt {attempt}/{max_retries}...")

        raw = llm.predict(
            CORRECTION_PROMPT,
            db_schema=schema_str,
            question=question,
            broken_sql=current_sql,
            error_message=current_error,
        )
        fixed_sql = extract_sql_from_llm_response(raw)
        print(f"  Fixed SQL: {fixed_sql}")

        try:
            result = execute_sql(db_engine, fixed_sql)
            attempts.append({"sql": fixed_sql, "success": True, "result": result})
            print(f"  Success! Result: {result}")
            return fixed_sql, result, True, attempts
        except Exception as e:
            current_error = str(e)
            current_sql = fixed_sql
            attempts.append({"sql": fixed_sql, "success": False, "error": current_error})
            print(f"  Still failing: {current_error}")

    return current_sql, None, False, attempts

print("Self-correction function defined.")


### Step 5.2 — Apply Self-Correction to Failed Candidates

We loop through our candidates. If any failed, we attempt to correct them.


In [ ]:
#@title Apply self-correction { display-mode: "form" }
corrected_candidates = []

for c in candidates:
    entry = dict(c)  # copy
    entry["correction_attempts"] = []

    if c["success"]:
        print(f"Strategy '{c['strategy']}': already succeeded — no correction needed.")
    else:
        print(f"\nStrategy '{c['strategy']}': FAILED — attempting self-correction...")
        fixed_sql, result, success, attempts = self_correct(
            llm, QUESTION, filtered_mschema_str, c["sql"], c["error"]
        )
        entry["sql"] = fixed_sql
        entry["result"] = result
        entry["success"] = success
        entry["error"] = None if success else entry.get("error")
        entry["correction_attempts"] = attempts
        entry["was_corrected"] = success

    corrected_candidates.append(entry)

print(f"\n{'='*60}")
ok = sum(c['success'] for c in corrected_candidates)
print(f"After self-correction: {ok}/{len(corrected_candidates)} candidates working")
print(f"{'='*60}")


### What just happened?

We implemented **XiYan-SQL's self-correction** mechanism:

| Step | What happens |
|------|-------------|
| Execute | Run the candidate SQL against the database |
| Detect | If it fails, capture the error message |
| Correct | Send error + SQL + question back to the LLM |
| Retry | Execute the corrected SQL (up to N times) |

**Key insight**: Error messages from the database engine are **precise feedback** — much better than asking the LLM to "check its work". This turns database errors into a learning signal.


---

## Section 6 — Candidate Selection: Picking the Best Answer

We now have multiple SQL candidates that (hopefully) all execute. But they might return **different results**.

How do we pick the best one? XiYan-SQL uses several strategies:

| Method | Idea |
|--------|------|
| **Majority vote** | If most candidates agree on the same result, that's probably correct |
| **LLM-as-judge** | Ask the LLM to evaluate which SQL best answers the question |

We'll implement both.


### Step 6.1 — Majority Vote

The simplest approach: convert each candidate's result to a string and count which result appears most often.


In [ ]:
#@title Majority vote function { display-mode: "form" }
from collections import Counter

def majority_vote(candidates):
    """Pick the result that appears most often among successful candidates."""
    results = []
    for c in candidates:
        if c["success"] and c["result"] is not None:
            results.append((str(c["result"]), c["strategy"]))

    if not results:
        return None, []

    counts = Counter(r[0] for r in results)
    winner_str, count = counts.most_common(1)[0]

    # Find which strategies agree
    agreeing = [s for r, s in results if r == winner_str]

    return winner_str, agreeing


winner_result, agreeing_strategies = majority_vote(corrected_candidates)

print(f"Question: {QUESTION}")
print(f"\nMajority vote result: {winner_result}")
print(f"Agreed by: {', '.join(agreeing_strategies)} ({len(agreeing_strategies)}/{len(corrected_candidates)})")

# Show all results for comparison
print(f"\n{'='*60}")
for c in corrected_candidates:
    status = '✓' if c['success'] else '✗'
    print(f"  {status} {c['strategy']:20s} → {c['result']}")


### Step 6.2 — LLM-as-Judge

For a more nuanced selection, we can ask the LLM to **evaluate** each candidate. This is especially useful when results differ and we need semantic judgement.


In [ ]:
JUDGE_PROMPT = PromptTemplate(
    """You are a SQL quality evaluator. Given a natural language question and several SQL candidate answers, pick the BEST one.

Database schema:
{db_schema}

Question: {question}

{candidates_block}

Evaluate each candidate on:
1. Correctness — does the SQL logically answer the question?
2. Completeness — does it handle edge cases (e.g. NULL values, filters)?
3. Clarity — is the SQL clean and readable?

Reply with:
- Your brief reasoning (2-3 sentences)
- Then on a new line: BEST: <strategy name>
""", prompt_type=PromptType.CUSTOM)


# Build the candidates block
cand_lines = []
for c in corrected_candidates:
    if c["success"]:
        cand_lines.append(f"--- Candidate: {c['strategy']} ---")
        cand_lines.append(f"SQL: {c['sql']}")
        cand_lines.append(f"Result: {c['result']}")
        cand_lines.append("")

candidates_block = "\n".join(cand_lines)

judge_response = llm.predict(
    JUDGE_PROMPT,
    db_schema=filtered_mschema_str,
    question=QUESTION,
    candidates_block=candidates_block,
)

print("LLM Judge says:")
print(judge_response)


### Step 6.3 — Final Results Dashboard


In [ ]:
#@title Visualization: Final Results Dashboard { display-mode: "form" }
import re as re_mod

# Extract the LLM's pick
best_match = re_mod.search(r'BEST:\s*(.*)', judge_response)
llm_pick = best_match.group(1).strip() if best_match else "Unknown"

# ── Visual: Final Dashboard ──

def render_final_dashboard(question, candidates, vote_result, vote_strategies, llm_pick):
    rows_html = ""
    for c in candidates:
        color = STRATEGY_COLORS.get(c['strategy'], '#64748b')
        status = '✓' if c['success'] else '✗'
        status_color = '#10b981' if c['success'] else '#ef4444'
        result_str = str(c['result']) if c['result'] is not None else 'N/A'

        badges = []
        if c['strategy'] in vote_strategies:
            badges.append('<span style="background:#dbeafe;color:#1e40af;padding:2px 8px;border-radius:10px;font-size:11px;margin-left:6px;">🗳 Majority</span>')
        if c['strategy'].lower() in llm_pick.lower() or llm_pick.lower() in c['strategy'].lower():
            badges.append('<span style="background:#fef3c7;color:#92400e;padding:2px 8px;border-radius:10px;font-size:11px;margin-left:6px;">⭐ LLM Pick</span>')
        if c.get('was_corrected'):
            badges.append('<span style="background:#fce7f3;color:#9d174d;padding:2px 8px;border-radius:10px;font-size:11px;margin-left:6px;">🔧 Corrected</span>')

        badge_html = ' '.join(badges)

        rows_html += f'''
        <tr style="border-bottom:1px solid #e2e8f0;">
            <td style="padding:12px;font-weight:600;color:{color};">{c['strategy']}{badge_html}</td>
            <td style="padding:12px;color:{status_color};font-weight:600;text-align:center;">{status}</td>
            <td style="padding:12px;font-family:monospace;font-size:12px;max-width:350px;word-break:break-all;">{c['sql']}</td>
            <td style="padding:12px;font-weight:700;text-align:center;">{result_str}</td>
        </tr>'''

    html = f'''
    <div style="font-family:'Segoe UI',system-ui,sans-serif;max-width:950px;margin:20px auto;">
        <div style="background:linear-gradient(135deg,#667eea 0%,#764ba2 100%);color:white;padding:24px 32px;border-radius:16px 16px 0 0;">
            <h2 style="margin:0 0 8px 0;">Final Results</h2>
            <div style="font-size:15px;opacity:0.9;">Question: \"{question}\"</div>
        </div>
        <div style="background:white;border:1px solid #e2e8f0;border-top:none;border-radius:0 0 16px 16px;overflow:hidden;">
            <table style="width:100%;border-collapse:collapse;">
                <thead>
                    <tr style="background:#f8fafc;border-bottom:2px solid #e2e8f0;">
                        <th style="padding:12px;text-align:left;color:#475569;">Strategy</th>
                        <th style="padding:12px;text-align:center;color:#475569;">Status</th>
                        <th style="padding:12px;text-align:left;color:#475569;">SQL</th>
                        <th style="padding:12px;text-align:center;color:#475569;">Result</th>
                    </tr>
                </thead>
                <tbody>{rows_html}</tbody>
            </table>
            <div style="padding:16px 20px;background:#f0fdf4;border-top:1px solid #e2e8f0;">
                <strong>🗳 Majority vote:</strong> {vote_result} (agreed by {', '.join(vote_strategies)})<br>
                <strong>⭐ LLM judge pick:</strong> {llm_pick}
            </div>
        </div>
    </div>'''
    display(HTML(html))


render_final_dashboard(QUESTION, corrected_candidates, winner_result, agreeing_strategies, llm_pick)


### What just happened?

We implemented **XiYan-SQL Stage 3** — candidate selection:

| Method | How it works | Strength |
|--------|-------------|----------|
| Majority vote | Count identical results across candidates | Simple, no extra LLM call |
| LLM-as-judge | Ask the LLM to reason about SQL quality | Can distinguish semantically equivalent but syntactically different queries |

**Key insight**: Having multiple candidates lets us apply **ensemble methods** — the same idea behind random forests and committee machines in classical ML.


---

## Section 7 — Try Your Own Questions!

Now let's put the entire pipeline together into one function and try different questions.


In [ ]:
#@title Full pipeline function { display-mode: "form" }
def text_to_sql_pipeline(question, schema_engine, llm, db_engine, max_corrections=2):
    """
    Complete Text-to-SQL pipeline:
      1. Keyword extraction -> schema filtering
      2. Multi-candidate generation (3 strategies)
      3. Self-correction for failed candidates
      4. Majority vote
    """
    import json as _json, re as _re
    print(f'\n{"="*70}')
    print(f'  QUESTION: {question}')
    print(f'{"="*70}')

    # -- Step 1: Keyword extraction --
    raw_kw = llm.predict(KEYWORD_EXTRACTION_PROMPT, question=question)
    try:
        m = _re.search(r'```json\s*(.*?)\s*```', raw_kw, _re.DOTALL)
        kws = _json.loads(m.group(1)) if m else _json.loads(raw_kw.strip())
    except Exception:
        kws = [w.strip().lower().strip('"[]') for w in raw_kw.split(',')]
    kws = [k.lower().strip() for k in kws if k.strip()]
    print(f'\n  Keywords: {kws}')

    # -- Step 2: Schema filtering --
    sel_cols = match_keywords_to_schema(schema_engine.mschema, kws)
    filt_schema = schema_engine.mschema.to_mschema(selected_columns=sel_cols)
    print(f'  Filtered to {len(sel_cols)} columns')

    # -- Step 3: Multi-candidate generation --
    evidence = build_evidence(schema_engine.mschema, sel_cols)
    strats = [
        ('Direct',           DIRECT_PROMPT,   {'db_schema': filt_schema, 'question': question}),
        ('Chain-of-Thought', COT_PROMPT,      {'db_schema': filt_schema, 'question': question}),
        ('With Evidence',    EVIDENCE_PROMPT, {'db_schema': filt_schema, 'question': question, 'evidence': evidence}),
    ]

    cands = []
    for name, tmpl, args in strats:
        raw = llm.predict(tmpl, **args)
        sql = extract_sql_from_llm_response(raw)
        try:
            result = execute_sql(db_engine, sql)
            cands.append({'strategy': name, 'sql': sql, 'result': result, 'success': True})
        except Exception as e:
            # -- Step 4: Self-correction --
            fixed_sql, result, ok, _ = self_correct(llm, question, filt_schema, sql, str(e), max_corrections)
            cands.append({'strategy': name, 'sql': fixed_sql, 'result': result, 'success': ok, 'was_corrected': ok})

    # -- Step 5: Selection --
    vote_result, vote_strats = majority_vote(cands)

    print(f'\n  Results:')
    for c in cands:
        s = 'OK' if c['success'] else 'FAIL'
        print(f'    {s} {c["strategy"]:20s} -> {c["result"]}')
    print(f'\n  Majority vote: {vote_result} (by {len(vote_strats)}/{len(cands)} strategies)')

    return vote_result, cands

print('Pipeline function defined.')


In [ ]:
#@title Run demo questions { display-mode: "form" }
# Try several questions!

DEMO_QUESTIONS = [
    "How many red papers are currently in stock?",
    "What is the total revenue from all delivered orders?",
    "Which customer has placed the most orders?",
    "List all refunds with the customer name and product name.",
    "What is the average price of cardstock products?",
]

all_results = {}
for q in DEMO_QUESTIONS:
    result, _ = text_to_sql_pipeline(q, schema_engine, llm, db_engine)
    all_results[q] = result


---

## Summary — The Complete XiYan-SQL Pipeline

Here is the full pipeline we implemented, mapped to the XiYan-SQL paper:

| Stage | Section | What it does | Key technique |
|-------|---------|-------------|---------------|
| **Schema enrichment** | §2 | Classify fields, generate descriptions → M-Schema | LLM-as-analyst |
| **Schema filtering** | §3 | Extract keywords, match against schema | LLM + keyword matching |
| **Multi-candidate SQL** | §4 | Generate SQL with 3 different prompt strategies | Prompt diversity |
| **Self-correction** | §5 | Feed errors back to LLM, retry | Error-driven feedback loop |
| **Candidate selection** | §6 | Majority vote + LLM-as-judge | Ensemble methods |

### Why this matters

Each stage addresses a different source of error:
- Schema enrichment → reduces **ambiguity** (the LLM understands what columns mean)
- Schema filtering → reduces **distraction** (the LLM only sees relevant tables)
- Multi-candidate → reduces **variance** (multiple shots increase hit rate)
- Self-correction → reduces **syntax errors** (database feedback fixes broken SQL)
- Selection → reduces **selection errors** (consensus beats any single candidate)

This is a great example of **agentic AI** — the LLM is not just called once, but orchestrated across multiple steps, with feedback loops and ensemble techniques, to achieve much higher accuracy than a single prompt could.
